In [8]:
import os

os.environ["CALITP_BQ_MAX_BYTES"] = str(400_000_000_000)

import datetime as dt  # noqa: E402
import sys  # noqa: E402
from functools import cache  # noqa: E402

from calitp_data_analysis.gcs_pandas import GCSPandas  # noqa: E402
from loguru import logger  # noqa: E402
from shared_utils import gtfs_utils_v2  # noqa: E402
from update_vars import COMPILED_CACHED_VIEWS, GTFS_DATA_DICT, SEGMENT_GCS  # noqa: E402
import pandas as pd

from shared_utils import DBSession, schedule_rt_utils  # noqa: E402

from calitp_data_analysis.gcs_geopandas import GCSGeoPandas  # noqa: E402s

@cache
def gcs_geopandas():
    return GCSGeoPandas()

# Fix Missing Santa Cruz Feed

A MTC 511 subfeed exists, but shouldn't be used. This situation is not considered in our current logic, which prefers subfeeds.

In [9]:
@cache
def gcs_pandas():
    return GCSPandas()


def get_operators(analysis_date: str):
    """
    Operators to download: favor subfeeds over combined regional feed.
    """
    all_operators = gtfs_utils_v2.schedule_daily_feed_to_gtfs_dataset_name(
        selected_date=analysis_date, keep_cols=None, get_df=True, feed_option="use_subfeeds"
    ).rename(columns={"gtfs_dataset_name": "name"})

    # patch until a better fix -- need to use a subfeed for Santa Cruz
    santa_cruz = gtfs_utils_v2.schedule_daily_feed_to_gtfs_dataset_name(
        selected_date=analysis_date, keep_cols=None, get_df=True, feed_option="include_precursor"
    ).rename(columns={"gtfs_dataset_name": "name"}).query('name.str.contains("Santa Cruz")')
    assert len(santa_cruz) == 1, 'query should return exactly one Santa Cruz schedule feed'

    keep_cols = ["feed_key", "name"]

    operators_to_include = pd.concat([all_operators, santa_cruz])[keep_cols]

    # There shouldn't be any duplicates by name, since we got rid
    # of precursor feeds. But, just in case, don't allow dup names.
    operators_to_include = operators_to_include.drop_duplicates(subset="name").reset_index(drop=True)

    return operators_to_include

In [10]:
from update_vars import analysis_date_list

In [11]:
df = get_operators(analysis_date_list[0])

/Users/edasmalchi/git/data-analyses/.venv/lib/python3.11/site-packages/google/cloud/bigquery/client.py:613: UserWarning: Cannot create BigQuery Storage client, the dependency google-cloud-bigquery-storage is not installed.
  warnings.warn(


In [12]:
df.query('name.str.contains("Union")') # missing in speeds but present here

,feed_key,name
39,fc2ea46aa5a43afe7cc24f44d8bd3eb4,Union City Flex
107,962a9feb664035676c5de19595fe6426,Bay Area 511 Union City Transit Schedule


In [13]:
df.query('name.str.contains("Santa Cruz")') # missing

,feed_key,name
239,84ff59aed2a4c130c2b4a59e89929d05,Santa Cruz Schedule


In [14]:
df = gtfs_utils_v2.schedule_daily_feed_to_gtfs_dataset_name(
    selected_date=analysis_date_list[0], keep_cols=None, get_df=True, feed_option="include_precursor"
).rename(columns={"gtfs_dataset_name": "name"})

In [15]:
df = df.query('name.str.contains("Santa Cruz")') # missing

In [16]:
len(df)

1

fix implemented by appending Santa Cruz for now

## Union City missing from speeds?

In [17]:
RAW_VP = GTFS_DATA_DICT.speeds_tables.raw_vp

In [18]:
f"{SEGMENT_GCS}{RAW_VP}_{analysis_date_list[0]}.parquet"

'gs://calitp-analytics-data/data-analyses/rt_segment_speeds/vp_2026-10-14.parquet'

In [21]:
gdf = gcs_geopandas().read_parquet(f"{SEGMENT_GCS}{RAW_VP}_{analysis_date_list[0]}.parquet")

In [22]:
gdf.query('gtfs_dataset_name.str.contains("Union")') 

,gtfs_dataset_name,schedule_gtfs_dataset_key,trip_id,trip_instance_key,location_timestamp,location_timestamp_local,gtfs_dataset_key,geometry
1101274,Bay Area 511 Union City Transit VehiclePositions,c95b7fcc18ea2e6b80c5c90a219d8592,14d8ee07-a69c-4c5e-9d75-994b28772910,91498283b3ebcd8dc16ea9b70be977bc,2026-10-14 22:42:42+00:00,2026-10-14 15:42:42,10a179fd35399c998ffa32b837a0ea9a,POINT (-122.00386 37.59122)
1101275,Bay Area 511 Union City Transit VehiclePositions,c95b7fcc18ea2e6b80c5c90a219d8592,14d8ee07-a69c-4c5e-9d75-994b28772910,91498283b3ebcd8dc16ea9b70be977bc,2026-10-14 22:25:15+00:00,2026-10-14 15:25:15,10a179fd35399c998ffa32b837a0ea9a,POINT (-122.04427 37.59347)
1101276,Bay Area 511 Union City Transit VehiclePositions,c95b7fcc18ea2e6b80c5c90a219d8592,14d8ee07-a69c-4c5e-9d75-994b28772910,91498283b3ebcd8dc16ea9b70be977bc,2026-10-14 22:42:11+00:00,2026-10-14 15:42:11,10a179fd35399c998ffa32b837a0ea9a,POINT (-122.00581 37.59333)
1101277,Bay Area 511 Union City Transit VehiclePositions,c95b7fcc18ea2e6b80c5c90a219d8592,14d8ee07-a69c-4c5e-9d75-994b28772910,91498283b3ebcd8dc16ea9b70be977bc,2026-10-14 22:32:34+00:00,2026-10-14 15:32:34,10a179fd35399c998ffa32b837a0ea9a,POINT (-122.01801 37.59076)
1101278,Bay Area 511 Union City Transit VehiclePositions,c95b7fcc18ea2e6b80c5c90a219d8592,14d8ee07-a69c-4c5e-9d75-994b28772910,91498283b3ebcd8dc16ea9b70be977bc,2026-10-14 22:16:22+00:00,2026-10-14 15:16:22,10a179fd35399c998ffa32b837a0ea9a,POINT (-122.06871 37.59258)
...,...,...,...,...,...,...,...,...
7675242,Union City Vehicle Positions,120943041e91041dc262b5852dd62ad8,f8405274-e312-4838-8052-0c243008787f,99fc7528c270eedc550c409b126161ac,2026-10-14 22:45:56+00:00,2026-10-14 15:45:56,c0ec4bad5229c854aa3fa2073bb03afe,POINT (-122.01787 37.59072)
7675243,Union City Vehicle Positions,120943041e91041dc262b5852dd62ad8,f9427dd6-084f-49af-b600-8cf68abb5eb9,5f02497a3b46e54177d59b34addc1527,2026-10-14 22:38:39+00:00,2026-10-14 15:38:39,c0ec4bad5229c854aa3fa2073bb03afe,POINT (-122.04233 37.59294)
7675244,Union City Vehicle Positions,120943041e91041dc262b5852dd62ad8,fba3a2e4-b110-42e4-bd3d-74ce2c13039f,978a036b3928b0d7d8aaa5d10ec4005b,2026-10-14 14:44:38+00:00,2026-10-14 07:44:38,c0ec4bad5229c854aa3fa2073bb03afe,POINT (-122.02987 37.60621)
7675245,Union City Vehicle Positions,120943041e91041dc262b5852dd62ad8,fba3a2e4-b110-42e4-bd3d-74ce2c13039f,978a036b3928b0d7d8aaa5d10ec4005b,2026-10-14 14:40:01+00:00,2026-10-14 07:40:01,c0ec4bad5229c854aa3fa2073bb03afe,POINT (-122.05091 37.60595)
